# Planogram Updater

Developed by Joel Vinas

Assisted by:
*   Google Antigravity
*   Google Gemini

In [2]:
!pip install diffusers transformers accelerate opencv-python Pillow

  Using cached click-8.3.2-py3-none-any.whl.metadata (2.6 kB)
Using cached click-8.3.2-py3-none-any.whl (108 kB)
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.2 which is incompatible.


In [3]:
!pip install --upgrade torch torchvision sympy

In [4]:
!pip install gTTS

  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
Using cached click-8.1.8-py3-none-any.whl (98 kB)
  Attempting uninstall: click
    Found existing installation: click 8.3.2
    Uninstalling click-8.3.2:
      Successfully uninstalled click-8.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [5]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the target output directory in Google Drive
# The provided URL 'https://drive.google.com/drive/folders/1juw87zkUDP_Wv_rOub9ETncr0fDI_O9r' points to a folder.
# We will use this folder ID to create a path in the mounted drive.
google_drive_output_folder_id = '1juw87zkUDP_Wv_rOub9ETncr0fDI_O9r'
google_drive_output_path = os.path.join('/content/drive/MyDrive', 'Planogram_Updates') # A more generic path within MyDrive

# Create the directory if it doesn't exist
os.makedirs(google_drive_output_path, exist_ok=True)

print(f"Google Drive output path set to: {google_drive_output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive output path set to: /content/drive/MyDrive/Planogram_Updates


In [10]:
from gtts import gTTS
import os

briefing_text = "The cold-medicine aisle is empty. I've updated the planogram to fill the top two shelves with our generic store-brand labels until the shipment arrives."
tts = gTTS(briefing_text)
tts.save(os.path.join(google_drive_output_path, "manager_briefing.mp3"))

In [7]:
# Uses T4 GPU
import os
import json
import torch
import cv2
import numpy as np
from PIL import Image
import urllib.request
from diffusers import StableDiffusionPipeline, StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [8]:
import os
# --- STEP A: Generate the Synthetic Pharmacy Base Layout ---
def generate_pharmacy_base(sd_pipe):
    print("Generating Synthetic Pharmacy Base Layout...")
    base_prompt = ("Interior photo of a modern pharmacy store, wide angle view of 5 aisles. "
                   "Focus on a central 4-row white metal shelf display with various health goods. "
                   "Red and white color scheme, bright lighting, photorealistic, 8k.")

    # Generate the base image using the standard Stable Diffusion pipeline
    base_image = sd_pipe(base_prompt, num_inference_steps=30, guidance_scale=8.5).images[0]
    if base_image is None:
        raise ValueError("Failed to generate base image. Check model loading and GPU availability.")
    print(f"Type of base_image after generation: {type(base_image)}")
    # Ensure google_drive_output_path is accessible, assuming it's defined globally after mounting
    base_image.save(os.path.join(google_drive_output_path, "pharmacy_base_layout.png"))
    print("Pharmacy Base Layout saved as 'pharmacy_base_layout.png'")
    return base_image

# --- STEP B: Update the Main Pipeline to use the Pharmacy Base ---
def main_pharmacy_scenario():
    # Initialize both Stable Diffusion and ControlNet pipelines
    sd_pipe, controlnet_pipe = setup_pipeline()

    # 1. Generate/Load the Pharmacy Base
    base_image = generate_pharmacy_base(sd_pipe)

    # 2. Extract Structure (ControlNet Constraint)
    # This 'locks' the 4-row shelf and 5 aisles into the AI's memory
    control_image = get_canny_edges(base_image)
    if control_image is None:
        raise ValueError("Failed to extract control image (Canny edges). Check base image validity.")
    print(f"Type of control_image after Canny: {type(control_image)}")

    # 3. Define the Pharmacy 'Disruption' Metadata
    pharmacy_metadata = {
        "room_type": "pharmacy aisle",
        "status": "Disrupted",
        "missing_items": ["Cough Medicine", "Cold Relief"],
        "substitutes": ["Store-Brand Healthcare Products", "Promotional Signage"]
    }

    # 4. Generate the 'Resilient' Update using the ControlNet pipeline
    structured_prompt = generate_structured_prompt(pharmacy_metadata)
    print(f"Updating Pharmacy Planogram with prompt: {structured_prompt}")

    updated_planogram = controlnet_pipe(
        structured_prompt,
        image=control_image,
        num_inference_steps=25,
        guidance_scale=7.5
    ).images[0]

    updated_planogram.save(os.path.join(google_drive_output_path, "pharmacy_updated_planogram.png"))
    print("Update complete. View 'pharmacy_updated_planogram.png' to see the resilient shelf.")

In [9]:
def get_canny_edges(image, low_threshold=100, high_threshold=200):
    image_np = np.array(image)
    image_np = cv2.Canny(image_np, low_threshold, high_threshold)
    image_np = image_np[:, :, None]
    image_np = np.concatenate([image_np, image_np, image_np], axis=2)
    return Image.fromarray(image_np)

# --- 2. Data-to-Prompt Engine ---
def generate_structured_prompt(metadata):
    """
    Maps JSON metadata into a Structured Prompt Template.
    """
    prompt = f"A high quality, highly detailed {metadata.get('room_type', 'room')}."

    if metadata.get('status') == 'Disrupted':
        prompt += " The layout shows some disruption. Needs restocking."
    else:
        prompt += " The layout is fully stocked and organized."

    substitutes = metadata.get('substitutes', [])
    if substitutes:
        prompt += f" Prominently features {', '.join(substitutes)} on the shelves."

    prompt += " Photorealistic, 8k resolution, award-winning interior design, bright lighting."
    return prompt

# --- 3. Stable Diffusion Pipeline Setup ---
def setup_pipeline():
    print("Loading Stable Diffusion and ControlNet models. This may take a moment...")
    # Standard Stable Diffusion Pipeline for base image generation
    sd_pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16
    )
    sd_pipe.scheduler = UniPCMultistepScheduler.from_config(sd_pipe.scheduler.config)
    sd_pipe.enable_model_cpu_offload()

    # ControlNet Pipeline for conditioned image generation
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16
    )
    controlnet_pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5", controlnet=controlnet, torch_dtype=torch.float16
    )
    controlnet_pipe.scheduler = UniPCMultistepScheduler.from_config(controlnet_pipe.scheduler.config)
    controlnet_pipe.enable_model_cpu_offload() # memory efficient for colab

    return sd_pipe, controlnet_pipe

# --- 4. Evaluation Module ---
def evaluate_generation(generated_image, prompt, expected_elements):
    """
    A framework for evaluating Prompt Alignment and Consistency.
    In practice, you would use a multimodal model like CLIP to compute similarities.
    We mock it here with deterministic-ish generation to fit the prompt structure.
    """
    # Dummy alignment metrics for report demonstration purposes
    alignment_score = max(0.0, min(1.0, 0.5 + (len(prompt) / 500.0) + np.random.uniform(-0.1, 0.1)))
    consistency_score = np.random.uniform(0.85, 0.98) # ControlNet ensures high structural consistency

    return {
        "Prompt Alignment": round(alignment_score, 3),
        "Consistency Score": round(consistency_score, 3)
    }

# --- 5. Main Execution Block ---
def main():
    main_pharmacy_scenario()

if __name__ == "__main__":
    main()

Loading Stable Diffusion and ControlNet models. This may take a moment...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

...

Generating Synthetic Pharmacy Base Layout...


  0%|          | 0/30 [00:00<?, ?it/s]

Type of base_image after generation: <class 'PIL.Image.Image'>
Pharmacy Base Layout saved as 'pharmacy_base_layout.png'
Type of control_image after Canny: <class 'PIL.Image.Image'>
Updating Pharmacy Planogram with prompt: A high quality, highly detailed pharmacy aisle. The layout shows some disruption. Needs restocking. Prominently features Store-Brand Healthcare Products, Promotional Signage on the shelves. Photorealistic, 8k resolution, award-winning interior design, bright lighting.


  0%|          | 0/25 [00:00<?, ?it/s]

Update complete. View 'pharmacy_updated_planogram.png' to see the resilient shelf.
